# 04 · Topluluk (Ensemble) Modelleri

| Yöntem | Fikir | Örnek |
|---|---|---|
| **Bagging** | Çok sayıda modeli paralel eğit, oyla/ortala → varyansı azaltır | Random Forest |
| **Boosting** | Modelleri sıralı eğit, her biri öncekinin hatasını düzeltir → bias'ı azaltır | AdaBoost, Gradient Boosting, XGBoost, LightGBM |
| **Stacking** | Farklı modellerin tahminlerini bir meta-model birleştirir | StackingClassifier |

> **Tablo (tabular) veride** genellikle en iyi performansı XGBoost / LightGBM / CatBoost verir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, recall_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, StackingClassifier, VotingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 1. Veri (Breast Cancer)

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)   # stratify: sınıf oranlarını koru

sonuclar = {}
def degerlendir(model, ad):
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    sonuclar[ad] = {"Accuracy": accuracy_score(y_test, p), "F1": f1_score(y_test, p),
                    "Recall": recall_score(y_test, p),
                    "ROC-AUC": roc_auc_score(y_test, proba) if proba is not None else np.nan}
    print(f"{ad}: " + "  ".join(f"{k}={v:.3f}" for k, v in sonuclar[ad].items()))
    return model

## 2. Baseline: tek karar ağacı

In [ ]:
degerlendir(DecisionTreeClassifier(random_state=42), "Tek Karar Ağacı")

**Yorum:** Sınırsız derinlikteki tek ağaç train verisini ezberler; ensemble'ların bu skoru ne kadar geçtiğine bakacağız.

## 3. Bagging – Random Forest

In [ ]:
rf = degerlendir(RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1), "Random Forest")

onem = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10)
onem.sort_values().plot.barh(figsize=(8, 4.5), color="tab:green")
plt.title("Random Forest – En Önemli 10 Özellik"); plt.tight_layout(); plt.show()

**Yorum:** Yüzlerce ağacın oylaması tek ağaçtaki varyansı büyük ölçüde azaltır. Özellik önemleri, tümörün boyut ve şekil düzensizliği (`worst` ön ekli) ölçümlerinin en belirleyici olduğunu gösterir.

In [ ]:
# Ağaç sayısı arttıkça performans
n_list = [1, 5, 10, 25, 50, 100, 200, 400]
skor = [f1_score(y_test, RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train).predict(X_test)) for n in n_list]
plt.figure(figsize=(7, 3.5)); plt.plot(n_list, skor, "o-", color="tab:green"); plt.xscale("log")
plt.title("Random Forest – Ağaç Sayısı vs Test F1"); plt.xlabel("n_estimators"); plt.show()

**Yorum:** Performans birkaç düzine ağaçtan sonra platoya ulaşır; daha fazla ağaç overfitting yapmaz, sadece hesaplama maliyetini artırır.

## 4. Boosting – AdaBoost, Gradient Boosting, XGBoost, LightGBM

In [ ]:
degerlendir(AdaBoostClassifier(n_estimators=200, random_state=42), "AdaBoost")
degerlendir(GradientBoostingClassifier(random_state=42), "Gradient Boosting")
xgb = degerlendir(XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                eval_metric="logloss", random_state=42), "XGBoost")
degerlendir(LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=42), "LightGBM")

**Önemli boosting hiperparametreleri:**

| Parametre | Etki |
|---|---|
| `n_estimators` | Ağaç sayısı (çok fazlası overfitting yapabilir — RF'nin aksine!) |
| `learning_rate` | Her ağacın katkısı; küçük değer + çok ağaç genelde daha iyi |
| `max_depth` | Ağaç derinliği (boosting'de 3–8 tipik) |
| `subsample`, `colsample_bytree` | Satır/sütun örnekleme → overfitting'i azaltır |

## 5. Voting ve Stacking

In [ ]:
taban = [("lr", Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))])),
         ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
         ("svm", Pipeline([("s", StandardScaler()), ("m", SVC(probability=True, random_state=42))]))]

degerlendir(VotingClassifier(estimators=taban, voting="soft"), "Voting (soft)")
degerlendir(StackingClassifier(estimators=taban, final_estimator=LogisticRegression(max_iter=1000)), "Stacking")

**Yorum:** Voting olasılıkların ortalamasını alır; Stacking ise taban modellerin tahminlerinden öğrenen bir meta-model kurar. Farklı türden (lineer + ağaç + çekirdek) modelleri birleştirmek, hataları birbirini telafi ettiği için daha sağlam sonuç verir.

## 6. Karşılaştırma

In [ ]:
karsilastirma = pd.DataFrame(sonuclar).T.sort_values("F1", ascending=False)
display(karsilastirma.round(3))
karsilastirma[["Accuracy", "F1", "ROC-AUC"]].plot.barh(figsize=(9, 5.5), xlim=(0.85, 1.0), colormap="Set2")
plt.title("Ensemble Modeller – Test Performansı"); plt.tight_layout(); plt.show()

## Özet
### Data Analysis Key Findings
- Tüm ensemble yöntemleri tek karar ağacına göre belirgin şekilde daha iyi sonuç verir (varyans azalması).
- Bu küçük ve neredeyse doğrusal ayrılabilir veri setinde ensemble'lar ile lojistik regresyon arasındaki fark küçüktür; büyük ve karmaşık tablo verilerinde XGBoost/LightGBM'in avantajı çok daha belirgin olur.
### Insights or Next Steps
- Boosting modellerinde `learning_rate`, `max_depth`, `n_estimators` için hiperparametre araması yapın (**10_Model_Secimi_ve_Hiperparametre**).